# FIREGUARD ID — 03: Model Design (Baseline ML: K-Means Clustering)
**Tujuan Minggu 3 (Bagian 1):** Melatih Model 1 (Baseline) dan menjustifikasinya berdasarkan teori, karakteristik data, *hypothesis space*, serta *inductive bias*, lalu mencatat hasil pada **Validation Set** (tanpa menyentuh Test Set).

---

## Bab 6.1.1 Teori & Alasan Pemilihan Model Baseline (K-Means)

### Teori Singkat
**K-Means Clustering** adalah algoritma *unsupervised partitioning* yang membagi $N$ observasi ke dalam $k$ klaster saling lepas dengan cara meminimalkan **Within-Cluster Sum of Squares (WCSS / Inertia)**:
$$\min_{C_1, \dots, C_k} \sum_{j=1}^{k} \sum_{x_i \in C_j} \|x_i - \mu_j\|_2^2$$
di mana $\mu_j$ adalah centroid (rata-rata vektor fitur) dari klaster $C_j$.

### Alasan Pemilihan Berdasarkan Karakteristik Data
1. **Dimensi Fitur Rendah & Numerik Kontinu:** Dataset grid $0{,}1^\circ$ Provinsi Riau ($N = 835$ total; $584$ Train, $125$ Validation) memiliki 5 fitur numerik kontinu (`hotspot_count`, `fire_days`, `mean_frp`, `max_frp`, `high_conf_count`) yang telah distandarisasi (`StandardScaler` di-*fit* hanya pada Train).
2. **Interpretabilitas Zonasi Risiko:** K-Means dengan $k=4$ menghasilkan 4 centroid yang mudah dipetakan langsung ke 4 tingkat kerawanan (*Rendah, Sedang, Tinggi, Sangat Tinggi*) untuk kebutuhan pengambilan keputusan mitigasi Karhutla.
3. **Komputasi Cepat & Reproducible:** Sebagai *baseline*, K-Means memberikan titik acuan metrik yang stabil dengan `random_state=42`.

## Bab 6.1.2 Hypothesis Space & Inductive Bias
- **Hypothesis Space:** Himpunan seluruh partisi ruang fitur terstandarisasi $\mathbb{R}^5$ yang dibentuk oleh $k$ titik centroid $\{\mu_1, \dots, \mu_k\}$, menghasilkan batas keputusan *piecewise-linear* berupa sel Voronoi.
- **Inductive Bias:**
  1. Mengasumsikan klaster berbentuk **sferis/konveks** dan berukuran/bervarians relatif seragam di ruang jarak Euclidean.
  2. Mengasumsikan seluruh titik data termasuk dalam salah satu klaster (*hard assignment*, tidak memodelkan *noise* atau *outlier* secara terpisah) — padahal data Karhutla Riau bersifat *right-skewed* sehingga titik ekstrem berpotensi menarik centroid.

## Bab 6.1.3–6.1.5 Pelatihan Baseline, Pencatatan Hyperparameter, & Hasil Validation Set

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import matplotlib.pyplot as plt
from data_loader import load_splits
from modeling import sweep_kmeans, train_kmeans
from evaluation import evaluate_clustering, compute_spatial_coherence

# Muat Train & Validation Set saja (Test Set tetap dikunci)
train_df, val_df, _ = load_splits('../data/split')
scaled_cols = [c for c in train_df.columns if c.endswith('_scaled')]
X_train = train_df[scaled_cols].values
X_val = val_df[scaled_cols].values

print(f'Train samples: {len(X_train)}, Validation samples: {len(X_val)}')

In [ ]:
# Sweep k = 2..8 untuk melihat kurva Elbow dan Silhouette pada Train & Validation
sweep_df = pd.DataFrame(sweep_kmeans(X_train, X_val, k_range=range(2, 9), random_state=42))
sweep_df

In [ ]:
# Pelatihan Model Baseline (k=4 sesuai kebutuhan 4 zona prioritas)
hyperparams_baseline = {'n_clusters': 4, 'init': 'k-means++', 'n_init': 10, 'random_state': 42}
print('Hyperparameter Baseline:', hyperparams_baseline)

km4, train_labels = train_kmeans(X_train, n_clusters=4, random_state=42)
val_labels = km4.predict(X_val)

train_metrics = evaluate_clustering(X_train, train_labels)
val_metrics = evaluate_clustering(X_val, val_labels)

val_tmp = val_df.copy()
val_tmp['cluster'] = val_labels
val_metrics['spatial_coherence'] = compute_spatial_coherence(val_tmp, label_col='cluster')

print('Hasil Train Set     :', train_metrics)
print('Hasil Validation Set:', val_metrics)